# **1. Perkenalan Dataset**

## Heart Disease Dataset

Dataset ini berasal dari **UCI Machine Learning Repository** dan tersedia di Kaggle:
- **Sumber**: https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset
- **Task**: Binary Classification (prediksi penyakit jantung)
- **Target**: `target` — 0 = tidak ada penyakit jantung, 1 = ada penyakit jantung

### Deskripsi Fitur:
| Fitur | Deskripsi |
|---|---|
| age | Usia pasien |
| sex | Jenis kelamin (1=pria, 0=wanita) |
| cp | Tipe nyeri dada (0-3) |
| trestbps | Tekanan darah saat istirahat (mm Hg) |
| chol | Kolesterol serum (mg/dl) |
| fbs | Gula darah puasa > 120 mg/dl (1=ya, 0=tidak) |
| restecg | Hasil EKG saat istirahat (0-2) |
| thalach | Detak jantung maksimum |
| exang | Angina akibat olahraga (1=ya, 0=tidak) |
| oldpeak | Depresi ST akibat olahraga |
| slope | Kemiringan segmen ST (0-2) |
| ca | Jumlah pembuluh darah besar (0-3) |
| thal | Thalassemia (0-3) |
| target | Diagnosis penyakit jantung (0=tidak, 1=ya) |

# **2. Import Library**

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

print('Library berhasil diimport!')
print(f'Pandas version: {pd.__version__}')
print(f'NumPy version: {np.__version__}')

Library berhasil diimport!
Pandas version: 2.2.3
NumPy version: 2.2.5


# **3. Memuat Dataset**

In [ ]:
# Load dataset
df = pd.read_csv('preprocessing/heart.csv')

print('Dataset berhasil dimuat!')
print(f'Shape: {df.shape}')
print(f'\n5 baris pertama:')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'preprocessing/heart.csv'

In [ ]:
# Informasi dasar dataset
print('=== INFO DATASET ===')
df.info()

In [ ]:
# Statistik deskriptif
print('=== STATISTIK DESKRIPTIF ===')
df.describe()

# **4. Exploratory Data Analysis (EDA)**

## 4.1 Pengecekan Missing Values

In [ ]:
# Cek missing values
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing Percentage (%)': missing_pct
})

print('=== MISSING VALUES ===')
print(missing_df)
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

## 4.2 Pengecekan Duplikasi

In [ ]:
# Cek duplikasi
duplicates = df.duplicated().sum()
print(f'Jumlah data duplikat: {duplicates}')
print(f'Persentase duplikat: {(duplicates/len(df))*100:.2f}%')

if duplicates > 0:
    print('\nContoh baris duplikat:')
    print(df[df.duplicated()])

## 4.3 Distribusi Target

In [ ]:
# Distribusi target
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
target_counts = df['target'].value_counts()
axes[0].bar(['No Disease (0)', 'Disease (1)'], target_counts.values, 
            color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0].set_title('Distribusi Target (Count)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Jumlah')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(target_counts.values, labels=['No Disease', 'Disease'], 
            colors=['#2ecc71', '#e74c3c'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Distribusi Target (Persentase)', fontsize=14, fontweight='bold')

plt.suptitle('Distribusi Kelas Target', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nDistribusi target:')
print(target_counts)
print(f'\nRasio kelas: {target_counts[0]/target_counts[1]:.2f}:1')

## 4.4 Distribusi Fitur Numerik

In [ ]:
# Distribusi fitur numerik
numerical_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    axes[i].hist(df[col], bins=30, color='#3498db', edgecolor='black', alpha=0.7)
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', label=f'Mean: {df[col].mean():.1f}')
    axes[i].axvline(df[col].median(), color='orange', linestyle='--', label=f'Median: {df[col].median():.1f}')
    axes[i].set_title(f'Distribusi {col}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frekuensi')
    axes[i].legend(fontsize=9)

axes[5].axis('off')
plt.suptitle('Distribusi Fitur Numerik', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('numerical_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.5 Distribusi Fitur Kategorikal

In [ ]:
# Fitur kategorikal
categorical_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']

for i, col in enumerate(categorical_cols):
    val_counts = df[col].value_counts()
    bar_colors = colors[:len(val_counts)]
    axes[i].bar(val_counts.index.astype(str), val_counts.values, 
                color=bar_colors, edgecolor='black')
    axes[i].set_title(f'Distribusi {col}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    for j, v in enumerate(val_counts.values):
        axes[i].text(j, v + 1, str(v), ha='center', fontweight='bold', fontsize=9)

plt.suptitle('Distribusi Fitur Kategorikal', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('categorical_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.6 Korelasi Antar Fitur

In [ ]:
# Heatmap korelasi
plt.figure(figsize=(14, 10))
corr_matrix = df.corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, cbar_kws={'shrink': 0.8},
            annot_kws={'size': 9})

plt.title('Heatmap Korelasi Antar Fitur', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Korelasi dengan target
print('=== KORELASI DENGAN TARGET ===')
target_corr = df.corr()['target'].sort_values(ascending=False)
print(target_corr)

## 4.7 Deteksi Outlier (Boxplot)

In [ ]:
# Boxplot untuk deteksi outlier
fig, axes = plt.subplots(1, 5, figsize=(20, 6))

for i, col in enumerate(numerical_cols):
    axes[i].boxplot(df[col], patch_artist=True,
                    boxprops=dict(facecolor='#3498db', alpha=0.7),
                    medianprops=dict(color='red', linewidth=2))
    axes[i].set_title(f'{col}', fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Nilai')
    
    # Hitung outlier dengan IQR
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
    axes[i].set_xlabel(f'Outlier: {len(outliers)}', fontsize=9, color='red')

plt.suptitle('Deteksi Outlier (Boxplot)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outlier_detection.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.8 Hubungan Fitur dengan Target

In [ ]:
# Fitur numerik vs target
fig, axes = plt.subplots(1, 5, figsize=(20, 6))

for i, col in enumerate(numerical_cols):
    data_0 = df[df['target']==0][col]
    data_1 = df[df['target']==1][col]
    
    axes[i].hist(data_0, bins=20, alpha=0.6, label='No Disease', color='#2ecc71', edgecolor='black')
    axes[i].hist(data_1, bins=20, alpha=0.6, label='Disease', color='#e74c3c', edgecolor='black')
    axes[i].set_title(f'{col} vs Target', fontsize=11, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frekuensi')
    axes[i].legend(fontsize=8)

plt.suptitle('Distribusi Fitur Numerik berdasarkan Target', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_vs_target.png', dpi=150, bbox_inches='tight')
plt.show()

# **5. Data Preprocessing**

## 5.1 Menghapus Data Duplikat

In [ ]:
print(f'Shape sebelum drop duplikat: {df.shape}')
df_clean = df.drop_duplicates()
print(f'Shape setelah drop duplikat: {df_clean.shape}')
print(f'Baris yang dihapus: {len(df) - len(df_clean)}')

df_clean = df_clean.reset_index(drop=True)
print('\nData duplikat berhasil dihapus!')

## 5.2 Menangani Missing Values

In [ ]:
print(f'Missing values sebelum penanganan:')
print(df_clean.isnull().sum())

# Isi missing values numerik dengan median, kategorikal dengan modus
numerical_cols_all = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
categorical_cols_all = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

for col in numerical_cols_all:
    if df_clean[col].isnull().sum() > 0:
        median_val = df_clean[col].median()
        df_clean[col].fillna(median_val, inplace=True)
        print(f'  {col}: diisi dengan median = {median_val}')

for col in categorical_cols_all:
    if df_clean[col].isnull().sum() > 0:
        mode_val = df_clean[col].mode()[0]
        df_clean[col].fillna(mode_val, inplace=True)
        print(f'  {col}: diisi dengan modus = {mode_val}')

print(f'\nMissing values setelah penanganan:')
print(df_clean.isnull().sum())

## 5.3 Encoding Fitur Kategorikal

In [ ]:
print('Shape sebelum encoding:', df_clean.shape)
print('\nFitur sebelum encoding:', list(df_clean.columns))

# One-Hot Encoding untuk fitur multi-kelas
multi_class_cols = ['cp', 'restecg', 'slope', 'thal']

df_encoded = pd.get_dummies(df_clean, columns=multi_class_cols, prefix=multi_class_cols, drop_first=False)

print('\nShape setelah encoding:', df_encoded.shape)
print('\nFitur setelah encoding:', list(df_encoded.columns))

## 5.4 Deteksi dan Penanganan Outlier (IQR Method)

In [ ]:
print('Shape sebelum penanganan outlier:', df_encoded.shape)

df_no_outlier = df_encoded.copy()

for col in numerical_cols_all:
    Q1 = df_no_outlier[col].quantile(0.25)
    Q3 = df_no_outlier[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outlier_count = ((df_no_outlier[col] < lower) | (df_no_outlier[col] > upper)).sum()
    
    # Clipping outlier ke batas IQR
    df_no_outlier[col] = df_no_outlier[col].clip(lower=lower, upper=upper)
    print(f'  {col}: {outlier_count} outlier di-clip ke [{lower:.2f}, {upper:.2f}]')

print('\nShape setelah penanganan outlier:', df_no_outlier.shape)

## 5.5 Normalisasi Fitur Numerik (StandardScaler)

In [ ]:
# Pisahkan fitur dan target
X = df_no_outlier.drop('target', axis=1)
y = df_no_outlier['target']

print(f'Shape X: {X.shape}')
print(f'Shape y: {y.shape}')
print(f'\nDistribusi target:')
print(y.value_counts())

# Normalisasi hanya untuk kolom numerik
scaler = StandardScaler()
X[numerical_cols_all] = scaler.fit_transform(X[numerical_cols_all])

print('\nStatistik setelah normalisasi:')
print(X[numerical_cols_all].describe().round(3))

## 5.6 Split Data (Train & Test)

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Shape X_train: {X_train.shape}')
print(f'Shape X_test:  {X_test.shape}')
print(f'Shape y_train: {y_train.shape}')
print(f'Shape y_test:  {y_test.shape}')
print(f'\nDistribusi y_train:')
print(y_train.value_counts())
print(f'\nDistribusi y_test:')
print(y_test.value_counts())

## 5.7 Simpan Dataset Hasil Preprocessing

In [ ]:
import os

# Gabungkan kembali X dan y untuk disimpan
df_preprocessed = X.copy()
df_preprocessed['target'] = y.values

# Simpan ke file CSV
output_path = 'heart-disease_preprocessing.csv'
df_preprocessed.to_csv(output_path, index=False)

print(f'Dataset preprocessing berhasil disimpan!')
print(f'File: {output_path}')
print(f'Shape: {df_preprocessed.shape}')
print(f'\nKolom:')
for col in df_preprocessed.columns:
    print(f'  - {col} ({df_preprocessed[col].dtype})')

## 5.8 Ringkasan Preprocessing

In [ ]:
print('=' * 50)
print('       RINGKASAN PREPROCESSING')
print('=' * 50)
print(f'Shape data awal      : {df.shape}')
print(f'Shape data akhir     : {df_preprocessed.shape}')
print(f'Duplikat dihapus     : {len(df) - len(df_clean)} baris')
print(f'Missing values       : Ditangani dengan median/modus')
print(f'Encoding             : One-Hot Encoding (cp, restecg, slope, thal)')
print(f'Outlier              : Clipping dengan metode IQR')
print(f'Normalisasi          : StandardScaler pada fitur numerik')
print(f'Split data           : 80% train, 20% test (stratified)')
print(f'Output               : heart-disease_preprocessing.csv')
print('=' * 50)